# Tiled assembly (template)

Single-mutant library for a CDS longer than one oligo: split into codon-aligned tiles, amplify each out of the pool with its own primer pair, and Golden Gate each into a destination vector holding the rest of the CDS. Fill in the input cell, then run top to bottom. A worked example is in `../tutorials/02-tiled-assembly.ipynb`.

In [ ]:
from library_designer import (
    LibrarySpec, CodonOptimizationParams, TiledAssemblyParams, SubstitutionScan,
)

# ===== EDIT =====
spec = LibrarySpec(
    name="my_tiled_library",
    protein_sequence="",              # your protein, one-letter, no stop codon
    substitutions=["A"],              # ["A"] alanine scan; list("ACDEFGHIKLMNPQRSTVWY") for full DMS
    cds=None,                         # native CDS to freeze verbatim; None to codon-optimize the protein
    optimization=CodonOptimizationParams(species="e_coli"),
    platform="twist_oligo_pools",
    tiled=TiledAssemblyParams(
        oligo_budget=300,             # hard cap on the final oligo length (bp)
        enzyme="BsaI",
        primer_set="subramanian2018", # bundled set, or a path to your own primer CSV
        # starting_vector="my_backbone.gb",  # point at your destination plasmid for full vector maps
    ),
    seed=0,
)
spec

## Build, tile, and check

In [ ]:
lib = SubstitutionScan(spec).generate().codon_optimize().tile()
print(lib.summary())   # per-tile primers, oligo lengths, and QC (junction sites, overhangs)

## Export

In [ ]:
import os

lib.drop_failed()
lib.to_oligo_pool("out/oligos.csv")       # the pooled synthesis order (name, sequence)
lib.to_primer_order("out/primers.csv")    # per-tile amplification primers (IDT bulk format)
lib.to_vectors("out/vectors.csv")         # per-tile destination vectors (manifest)
lib.to_design_specs(f"out/{lib.spec.name}_design_specs.json")
# With tiled.starting_vector set, also emit annotated plasmid maps (needs the `tiled` extra):
# lib.to_vector_maps("out/vectors/")

print("wrote files to", os.path.abspath("out"))
for root, _, files in os.walk("out"):
    for f in sorted(files):
        print("  ", os.path.join(root, f))